# 02 · BM25: evaluation functions and baseline B1

Runs on **Kaggle CPU**, no internet needed, in about 5–10 minutes.

**Inputs:** the dataset `wm-src` (the repo's `src/`, see README); Add Input → Notebook Output → `01_preprocess` (`pairs.parquet` and `products_clean.parquet`).

**Output** (`/kaggle/working`):

| File | Content | In git |
|---|---|:---:|
| `results/<run>.json` | Configuration and metrics on val and test | yes |
| `results/scores/<run>_{val,test}.parquet` | `example_id, query_id, score` | yes |
| `bm25_features.parquet` | `example_id, bm25_title, bm25_main` for **all** splits, used as features in `05_oof_lgbm` | no |

**Runs:** `b0_random` (constant score, i.e. random order), `b1_bm25_title`, `b1_bm25_main`.

**Reference values (from the EDA):** a constant score gives nDCG ≈ 0.742 on val and 0.747 on test; BM25 on titles gives ≈ 0.816 on val.

In [ ]:
# Setup: make src/ importable. On Kaggle, attach the dataset `wm-src` (zip of src/, see README).
import glob, sys
from pathlib import Path

_hits = glob.glob('/kaggle/input/**/src/evaluation.py', recursive=True) + glob.glob('../src/evaluation.py')
assert _hits, 'src/ not found: Add Input -> dataset wm-src (zip of src/, see README)'
ROOT = Path(_hits[0]).resolve().parents[1]
sys.path.insert(0, str(ROOT))

from src.evaluation import src_version
print('src from', ROOT, '|', src_version())

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd

from src.data import PREPROCESS_HINT, find
from src.evaluation import RunLog, summarize
from src.features import BM25, TOKEN

SEED = 42
OUT = Path('/kaggle/working')
RES = OUT / 'results'

pairs = pd.read_parquet(find('pairs.parquet', PREPROCESS_HINT))
prod = pd.read_parquet(find('products_clean.parquet', PREPROCESS_HINT),
                       columns=['product_id', 'title', 'brand', 'color', 'bullets'])
print('pairs', pairs.shape, '| products', prod.shape)
print('pairs per split:', pairs['split'].value_counts().to_dict())

## 1. Evaluation functions

Every notebook scores its runs with `src/evaluation.py`, so all result tables are computed the same way.

| Metric | Meaning |
|---|---|
| `ndcg` | nDCG over the full candidate list, official gains E 1 / S 0.1 / C 0.01 / I 0. **Main metric.** |
| `ndcg_amazon` | As `ndcg`, with the S and C gains swapped as in Amazon's script. Only for comparison with the 0.857 reference |
| `ndcg10` | nDCG@10, official gains |
| `mrr_e` | 1 / rank of the first E product; only queries with at least one E |
| `e_over_s` | Share of (E, S) pairs within a query where E scores higher; ties count 0.5; pooled over all queries |

- Ties are broken at random, and every metric is the mean over 3 seeds.
- Queries whose candidates are all I have IDCG = 0 and are left out of the means; their number is in `n_skipped_all_irrelevant`.
- 95% confidence intervals come from a bootstrap over queries (1,000 resamples). Two runs on the same queries are compared with a paired bootstrap.

Quick checks of the functions: using the gain itself as the score must give nDCG = 1; the negated gain must score below random; a constant score must give E-over-S = 0.5.

In [3]:
MASK = {s: (pairs['split'] == s).to_numpy() for s in ['val', 'test']}
val = pairs[MASK['val']]

oracle, _ = summarize(val, val['gain'])
worst, _ = summarize(val, -val['gain'])
flat, _ = summarize(val, np.zeros(len(val)))
print('oracle', oracle, '\nworst ', worst, '\nflat  ', flat)
assert oracle['ndcg'] == 1.0 and oracle['ndcg10'] == 1.0 and oracle['mrr_e'] == 1.0 and oracle['e_over_s'] == 1.0
assert worst['ndcg'] < flat['ndcg'] < oracle['ndcg']
assert flat['e_over_s'] == 0.5

oracle {'ndcg': 1.0, 'ndcg_amazon': 0.9983, 'ndcg10': 1.0, 'mrr_e': 1.0, 'ndcg_ci95': [1.0, 1.0], 'e_over_s': 1.0, 'n_queries': 2089, 'n_skipped_all_irrelevant': 0} 
worst  {'ndcg': 0.56, 'ndcg_amazon': 0.5194, 'ndcg10': 0.2056, 'mrr_e': 0.1142, 'ndcg_ci95': [0.554, 0.5656], 'e_over_s': 0.0, 'n_queries': 2089, 'n_skipped_all_irrelevant': 0} 
flat   {'ndcg': 0.7422, 'ndcg_amazon': 0.709, 'ndcg10': 0.5749, 'mrr_e': 0.6063, 'ndcg_ci95': [0.7361, 0.7485], 'e_over_s': 0.5, 'n_queries': 2089, 'n_skipped_all_irrelevant': 0}


## 2. Logging a run

`RunLog` (`src/evaluation.py`) writes `results/<run>.json` and `results/scores/<run>_{val,test}.parquet`; the json also records `src_version` and library versions. Every notebook that produces scores uses this format, so `06_report` can read them all.

In [ ]:
log = RunLog(RES, {s: pairs[m] for s, m in MASK.items()})


def log_run(name, score_all, info, compare_to=None):
    # score_all: one score per row of `pairs` (all splits, same order)
    score_all = np.asarray(score_all, dtype=float)
    return log.log(name, {s: score_all[m] for s, m in MASK.items()}, info, compare_to=compare_to)

## 3. B0: random order

Every product gets the same score, so the order is decided entirely by the random tie-break. Every model must beat this floor.

In [5]:
rec = log_run('b0_random', np.zeros(len(pairs)), {'model': 'constant score (random order)'})
assert abs(rec['metrics']['val']['ndcg'] - 0.742) < 0.005 and abs(rec['metrics']['test']['ndcg'] - 0.747) < 0.005

b0_random {'val': (0.7422, [0.7361, 0.7485]), 'test': (0.7459, [0.7429, 0.7491])}


## 4. B1: BM25

- `BM25` and `tokens` are in `src/features.py` (also used for the Direction 3 features).
- Okapi BM25 with `k1 = 1.2`, `b = 0.75`. IDF and average document length are computed over the **whole catalogue** (482,105 products), not within each candidate list.
- Tokenisation as in the EDA: lowercase, `[a-z0-9]+` tokens, decimals such as `10.2` kept. Each query term counts once.
- Two fields:
  - `title`;
  - `main` = title + brand + color + bullets, joined with spaces. The `text_main` column is not used, because its `brand:` and `color:` labels would become fake terms present in every product.
- BM25 does not learn from labels, so it can score every split and be saved as a LightGBM feature.

In [ ]:
FIELDS = {
    'title': prod['title'],
    'main': prod['title'] + ' ' + prod['brand'].fillna('') + ' ' + prod['color'].fillna('') + ' ' + prod['bullets'].fillna(''),
}

feats = pairs[['example_id']].copy()
for field, docs in FIELDS.items():
    t0 = time.time()
    model = BM25(docs.tolist())
    text = pairs['product_id'].map(pd.Series(docs.to_numpy(), index=prod['product_id']))
    feats[f'bm25_{field}'] = [model.score(q, d) for q, d in zip(pairs['query'], text)]
    print(f'{field}: vocab {len(model.df):,}, avgdl {model.avgdl:.1f}, {time.time() - t0:.0f} s')

feats.describe().round(2)

In [7]:
info = {'model': 'BM25', 'k1': 1.2, 'b': 0.75, 'idf': 'whole catalogue', 'tokenizer': TOKEN.pattern}
rec = log_run('b1_bm25_title', feats['bm25_title'], {**info, 'fields': 'title'}, compare_to='b0_random')
assert abs(rec['metrics']['val']['ndcg'] - 0.816) < 0.01, 'BM25 title far from the EDA value 0.816'
log_run('b1_bm25_main', feats['bm25_main'], {**info, 'fields': 'title + brand + color + bullets'},
        compare_to='b1_bm25_title');

b1_bm25_title {'val': (0.8162, [0.8096, 0.8227]), 'test': (0.8209, [0.8179, 0.824])}
b1_bm25_main {'val': (0.8227, [0.8165, 0.8293]), 'test': (0.8244, [0.8209, 0.8275])}


## 5. Comparison table

`paired_vs` is the nDCG difference between a run and the run it is compared to, over the same queries. If `ci95` excludes 0, the difference is statistically significant.

In [ ]:
display(log.table())
log.print_paired()

nDCG on test by query type (`source`), to see where BM25 is strong or weak. For example, queries with negation (`negations`) are hard for BM25, because it rewards the negated term too.

In [ ]:
src = pairs.drop_duplicates('query_id').set_index('query_id')['source']
by_src = pd.DataFrame({n: log.per_query[n]['test'].set_index('query_id')['ndcg'] for n in log.records})
by_src = by_src.join(src).groupby('source').agg(['mean', 'size']).round(4)
by_src

## 6. Save the BM25 features for all splits

In [10]:
feats.to_parquet(OUT / 'bm25_features.parquet', index=False)
for f in sorted(p for p in OUT.rglob('*') if p.is_file()):
    print(f'{str(f.relative_to(OUT)):42s} {f.stat().st_size / 1e6:7.2f} MB')

__notebook__.ipynb                            0.04 MB
bm25_features.parquet                        10.83 MB
results/b0_random.json                        0.00 MB
results/b1_bm25_main.json                     0.00 MB
results/b1_bm25_title.json                    0.00 MB
results/scores/b0_random_test.parquet         1.07 MB
results/scores/b0_random_val.parquet          0.27 MB
results/scores/b1_bm25_main_test.parquet      2.03 MB
results/scores/b1_bm25_main_val.parquet       0.48 MB
results/scores/b1_bm25_title_test.parquet     1.83 MB
results/scores/b1_bm25_title_val.parquet      0.44 MB
